In [ ]:
import kagglehub
import pandas as pd
import os
# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")

path = os.path.join(path, 'Q3_data.csv')

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()



In [ ]:
# Task 1: Write your code here:
# saving target for later
y = df['Target'].copy()
df_feat = df.drop('Target', axis = 1).copy()

for col in df_feat.columns:
  null_count = df_feat[col].isnull().sum()/len(df_feat[col])
  #print(null_count)
  if null_count > 0.7:
    df_feat = df_feat.drop(columns = col)

for col in df_feat.select_dtypes(exclude = ['object']).columns:
  if df_feat[col].median() == 0:
    df_feat = df_feat.drop(columns = col)


for col in df_feat.select_dtypes(exclude = ['object']).columns:
  df_feat[col] = df_feat[col].fillna(df_feat[col].median())

df_feat.head()

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_feat)

In [ ]:
# Task 3: Write your code here:

# checking for categorical data
print(df_feat.select_dtypes(include=['object']).columns)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

for col in df_feat.columns:
  scaler = StandardScaler()
  df_feat[col] = scaler.fit_transform(df_feat[[col]])

df_feat.head()

In [ ]:
# Task 5: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df_feat
y = df['Target']

y.head()

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score

from catboost import CatBoostClassifier

f1_error = []
accuracy_list = []

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_index, test_index in kf.split(X, y):
  model = CatBoostClassifier(verbose=0,
      n_estimators=200,
      max_depth=5)

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  f1 = f1_score(y_test, y_pred)
  acc = accuracy_score(y_test, y_pred)

  print(f"Accuracy: {acc :.5f} f1-score: {f1 :.5f}")

  f1_error.append(f1)
  accuracy_list.append(acc)


In [ ]:
# Task 1: Write your code here:
import numpy as np
import matplotlib.pyplot as plt

importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
print(f"Golden feature: {importance['feature'][0]}")

In [ ]:
# Task Bonus: Write your code here:
X = df_feat['P_2']
